# 🌦️ Climate Analyzer — v2
**Autor:** Ariel Macías | Agrónomo · GIS & Remote Sensing

Análisis climático sobre un lote agrícola usando **ERA5-Land** (ECMWF) via Google Earth Engine.

| Celda | Descripción |
|-------|-------------|
| 0 | Configuración (editar `config.py`) |
| 1 | Inicializar GEE y dibujar lote |
| 2 | Descarga ERA5 + métricas agronómicas + gráficos |

### Variables extraídas de ERA5-Land
| Variable | Descripción |
|----------|-------------|
| Precipitación | Suma diaria (mm) |
| T. máx / mín / media | Temperatura a 2 m (°C) |
| Punto de rocío | → Humedad Relativa estimada (%) |
| Radiación solar | Descendente superficial (MJ/m²/día) |
| Viento (u, v) | Componentes a 10 m → velocidad escalar (m/s) |
| ET vegetación | Evapotranspiración real ERA5 (mm) |
| **ETo Penman-Monteith** | Calculada con FAO-56 a partir de las anteriores |
| **Balance hídrico** | Lluvia acum. − ETo acum. (mm) |
| **GDA** | Grados día acumulados sobre T_BASE (config.py) |

In [ ]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# Editá config.py para cambiar proyecto GEE, T_BASE y centro del mapa.
# ============================================================
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from config import GEE_PROJECT, CENTRO_LAT, CENTRO_LON, ZOOM_INICIAL, T_BASE

print('✅ Configuración cargada')
print(f'   Proyecto GEE : {GEE_PROJECT}')
print(f'   Centro mapa  : ({CENTRO_LAT}, {CENTRO_LON})')
print(f'   T. base GDA  : {T_BASE} °C')

---
## Paso 1 · Inicializar GEE y dibujar el lote
Usá la herramienta de polígono (panel izquierdo) para delimitar el lote. Luego ejecutá la Celda 2.

In [ ]:
# ============================================================
# CELDA 1 — INICIALIZACIÓN GEE Y DIBUJO DEL LOTE
# ============================================================
import ee
import geemap
from IPython.display import display
from gee_utils import inicializar_gee

inicializar_gee('your_project')

print('\n--- PASO 1: DIBUJÁ TU LOTE ---')
print('Usá el polígono de la izquierda para dibujar. Cuando termines, ejecutá la Celda 2.')

Draw_Map = geemap.Map(center=[CENTRO_LAT, CENTRO_LON], zoom=ZOOM_INICIAL)
Draw_Map.add_basemap('SATELLITE')
display(Draw_Map)

---
## Paso 2 · Descarga ERA5 · Métricas agronómicas · Gráficos
Descarga datos ERA5-Land para el lote dibujado, calcula ETo (Penman-Monteith FAO-56),
balance hídrico y grados día acumulados, y genera los paneles interactivos.

In [ ]:
# ============================================================
# CELDA 2 — ANÁLISIS COMPLETO
# ============================================================
from shapely.geometry import shape
import pandas as pd

from gee_utils      import obtener_fecha_maxima, calcular_periodos, descargar_serie
from agro_metrics   import calcular_eto_pm, calcular_hr, agregar_mensual, procesar_diario
from plots          import mostrar_graficos
from config         import ERA5_COLLECTION, T_BASE

# 1. Verificar dibujo
if Draw_Map.user_roi is None:
    raise ValueError('⚠️ No se detectó ningún dibujo. Volvé a la Celda 1.')

# 2. Geometría del lote (área completa, no centroide)
roi_info  = Draw_Map.user_roi.getInfo()
lote_geom = ee.Geometry(roi_info.get('geometry', roi_info))
centroid  = lote_geom.centroid(maxError=1).coordinates().getInfo()
latitud   = centroid[1]   # necesaria para ETo PM

# 3. Fechas automáticas
era5_base = ee.ImageCollection(ERA5_COLLECTION).filterBounds(lote_geom)
fecha_max = obtener_fecha_maxima(era5_base)
hist_start, hist_end, daily_start, daily_end = calcular_periodos(fecha_max)

print(f'📅 Histórico  : {hist_start} → {hist_end}')
print(f'📅 Mes actual : {daily_start} → {daily_end}')

# 4. Descarga ERA5 (area mean sobre el lote, no centroide)
df = descargar_serie(lote_geom, hist_start, hist_end)

# 5. Métricas agronómicas
df['eto'] = calcular_eto_pm(df, latitud)
df['hr']  = calcular_hr(df)

# 6. Agregación mensual
df_mensual = agregar_mensual(df)

# 7. Detalle diario del mes actual
df_diario = procesar_diario(df, daily_start, daily_end)
df_diario.attrs['t_base'] = T_BASE   # para label del gráfico

# 8. Métricas resumen (imprimir antes de los gráficos)
mes_actual = df_mensual.iloc[-1]
print(f'\n📊 Resumen del último mes disponible ({mes_actual["mes_año"].strftime("%b %Y")})')
print(f'   Lluvia      : {mes_actual["precip"]:.1f} mm')
print(f'   ETo PM      : {mes_actual["eto"]:.1f} mm')
print(f'   Balance     : {mes_actual["balance_hidro"]:+.1f} mm')
print(f'   T. media    : {mes_actual["t_med"]:.1f} °C')
print(f'   HR media    : {mes_actual["hr"]:.0f} %')

# 9. Gráficos
print('\n📈 Generando gráficos interactivos...')
mostrar_graficos(df_mensual, df_diario, daily_start, daily_end)